In [1]:
# ============================================================
# DAY 56/100
# PROJECT: CLOUD-READY LANGCHAIN AI ASSISTANT
# PART 1: LANGCHAIN CORE APPLICATION
#
# JUPYTER NOTEBOOK ONLY
#
# HARDWARE OPTIMIZED:
# - SLOW CPU
# - VERY LIMITED STORAGE
# - NO GPU
# - NO LARGE LLM
# - NO EXTERNAL DATASET
# ============================================================


# ============================================================
# INSTALL MINIMAL PACKAGES
# ============================================================

!pip install -q langchain langchain-core


# ============================================================
# IMPORTS
# ============================================================

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

import time


print("LangChain imported successfully ✔")


# ============================================================
# PROJECT INFORMATION
# ============================================================

PROJECT_NAME = "Cloud-Ready LangChain AI Assistant"

PROJECT_VERSION = "1.0"

DAY = "Day 56/100"

print()
print("=" * 60)
print(PROJECT_NAME)
print(DAY)
print("=" * 60)


# ============================================================
# CREATE A SMALL LOCAL KNOWLEDGE BASE
#
# We intentionally create the knowledge base locally.
# This avoids downloading a dataset.
# ============================================================

documents = [

    Document(
        page_content="""
        LangChain is a framework for building applications
        powered by language models. It provides components
        for prompts, document processing, retrieval, tools,
        agents, and application workflows.
        """,
        metadata={
            "topic": "LangChain"
        }
    ),

    Document(
        page_content="""
        Retrieval-Augmented Generation, commonly called RAG,
        combines information retrieval with language models.
        A RAG system retrieves relevant information from a
        knowledge source and uses that information to generate
        a response.
        """,
        metadata={
            "topic": "RAG"
        }
    ),

    Document(
        page_content="""
        AI agents are systems that can decide which actions
        or tools should be used to solve a task. Agents can
        interact with tools, observe results, and continue
        processing until the task is completed.
        """,
        metadata={
            "topic": "AI Agents"
        }
    ),

    Document(
        page_content="""
        FastAPI is a Python framework commonly used to build
        APIs. It can expose an AI application through HTTP
        endpoints such as a chat endpoint.
        """,
        metadata={
            "topic": "FastAPI"
        }
    ),

    Document(
        page_content="""
        Docker packages an application and its dependencies
        into a container. Containers help applications run
        consistently across development, testing, and
        production environments.
        """,
        metadata={
            "topic": "Docker"
        }
    ),

    Document(
        page_content="""
        Cloud deployment allows an application to run on
        remote infrastructure instead of only on a local
        computer. AI applications can expose APIs and services
        through cloud infrastructure.
        """,
        metadata={
            "topic": "Cloud"
        }
    ),

    Document(
        page_content="""
        Production AI applications should consider reliability,
        logging, monitoring, error handling, security,
        scalability, and performance.
        """,
        metadata={
            "topic": "Production AI"
        }
    )

]


print()
print("Knowledge base created ✔")
print("Number of documents:", len(documents))


# ============================================================
# DISPLAY KNOWLEDGE BASE
# ============================================================

print()
print("=" * 60)
print("KNOWLEDGE BASE")
print("=" * 60)


for index, document in enumerate(documents):

    print()
    print("Document:", index + 1)

    print(
        "Topic:",
        document.metadata["topic"]
    )

    print(
        "Content:",
        document.page_content.strip()
    )


# ============================================================
# SIMPLE LIGHTWEIGHT RETRIEVER
#
# We are intentionally NOT using embeddings.
# This keeps the project extremely lightweight.
#
# The retriever performs simple keyword matching.
# ============================================================

def retrieve_documents(
    query,
    top_k=3
):

    query_words = set(
        query.lower().split()
    )


    scored_documents = []


    for index, document in enumerate(documents):

        text = document.page_content.lower()


        score = 0


        for word in query_words:

            clean_word = (
                word
                .replace("?", "")
                .replace(",", "")
                .replace(".", "")
                .replace("!", "")
            )


            if len(clean_word) < 3:

                continue


            if clean_word in text:

                score += 1


        if score > 0:

            scored_documents.append(

                (
                    score,
                    index,
                    document
                )

            )


    # Sort by highest relevance score

    scored_documents.sort(
        key=lambda item: item[0],
        reverse=True
    )


    # Return top documents

    results = scored_documents[
        :top_k
    ]


    return results


# ============================================================
# TEST RETRIEVER
# ============================================================

test_query = (
    "What is LangChain?"
)


retrieved = retrieve_documents(
    test_query
)


print()
print("=" * 60)
print("RETRIEVER TEST")
print("=" * 60)

print()
print("Query:")
print(test_query)

print()
print("Retrieved Documents:")


for score, index, document in retrieved:

    print()
    print(
        "Document:",
        index + 1
    )

    print(
        "Score:",
        score
    )

    print(
        "Topic:",
        document.metadata["topic"]
    )


# ============================================================
# CREATE CONTEXT FROM RETRIEVED DOCUMENTS
# ============================================================

def create_context(
    retrieved_documents
):

    context_parts = []


    for score, index, document in retrieved_documents:

        topic = document.metadata[
            "topic"
        ]


        content = document.page_content.strip()


        context_parts.append(

            "Topic: "
            + topic
            + "\n"
            + content

        )


    return "\n\n".join(
        context_parts
    )


# ============================================================
# TEST CONTEXT CREATION
# ============================================================

context = create_context(
    retrieved
)


print()
print("=" * 60)
print("RETRIEVED CONTEXT")
print("=" * 60)

print()
print(context)


# ============================================================
# CREATE LANGCHAIN PROMPT TEMPLATE
# ============================================================

prompt = ChatPromptTemplate.from_messages(

    [

        (
            "system",

            """
            You are a helpful AI Engineering assistant.

            Answer questions using the provided context.

            If the context contains the answer, explain it
            clearly and simply.

            If the context does not contain enough information,
            say that the information is not available in the
            knowledge base.

            Keep answers concise and useful.

            CONTEXT:
            {context}
            """
        ),

        (
            "human",

            "{question}"

        )

    ]

)


print()
print("=" * 60)
print("LANGCHAIN PROMPT CREATED")
print("=" * 60)

print()
print("Prompt template created successfully ✔")


# ============================================================
# LIGHTWEIGHT RESPONSE GENERATOR
#
# Since we cannot use a large LLM on your machine, this
# function creates a deterministic response from the retrieved
# knowledge.
#
# In a cloud production system, this layer can be replaced
# by an API-based LLM without changing the retrieval design.
# ============================================================

def generate_response(
    question,
    context
):

    if context.strip() == "":

        return (
            "I could not find relevant information "
            "in the knowledge base."
        )


    # --------------------------------------------------------
    # Find the strongest matching document
    # --------------------------------------------------------

    retrieved_documents = retrieve_documents(
        question,
        top_k=1
    )


    if len(retrieved_documents) == 0:

        return (
            "I could not find relevant information "
            "in the knowledge base."
        )


    score, index, document = (
        retrieved_documents[0]
    )


    topic = document.metadata[
        "topic"
    ]


    content = document.page_content.strip()


    response = (

        "Based on the knowledge base:\n\n"

        + content

        + "\n\n"

        + "Source topic: "
        + topic

    )


    return response


# ============================================================
# CREATE LANGCHAIN RESPONSE CHAIN
#
# Architecture:
#
# QUESTION
#     ↓
# RETRIEVER
#     ↓
# CONTEXT
#     ↓
# PROMPT
#     ↓
# RESPONSE
# ============================================================

def retrieve_for_chain(
    question
):

    retrieved_documents = retrieve_documents(
        question,
        top_k=3
    )


    context = create_context(
        retrieved_documents
    )


    return context


retrieval_chain = RunnableLambda(
    retrieve_for_chain
)


response_chain = RunnableLambda(

    lambda context_and_question:

        generate_response(

            context_and_question["question"],

            context_and_question["context"]

        )

)


# ============================================================
# CREATE COMPLETE LANGCHAIN PIPELINE
# ============================================================

def langchain_assistant(
    question
):

    start_time = time.time()


    # --------------------------------------------------------
    # RETRIEVAL
    # --------------------------------------------------------

    context = retrieval_chain.invoke(
        question
    )


    # --------------------------------------------------------
    # RESPONSE
    # --------------------------------------------------------

    answer = response_chain.invoke(

        {

            "question":
            question,

            "context":
            context

        }

    )


    # --------------------------------------------------------
    # LATENCY
    # --------------------------------------------------------

    latency = (
        time.time()
        - start_time
    )


    return {

        "question":
        question,

        "context":
        context,

        "answer":
        answer,

        "latency":
        latency

    }


# ============================================================
# TEST COMPLETE LANGCHAIN ASSISTANT
# ============================================================

test_questions = [

    "What is LangChain?",

    "What is RAG?",

    "What are AI agents?",

    "What is FastAPI?",

    "What is Docker?",

    "What is cloud deployment?"

]


print()
print("=" * 60)
print("LANGCHAIN ASSISTANT TEST")
print("=" * 60)


for question in test_questions:

    result = langchain_assistant(
        question
    )


    print()
    print("-" * 60)

    print(
        "USER:",
        question
    )

    print()

    print(
        "ASSISTANT:"
    )

    print(
        result["answer"]
    )

    print()

    print(
        "Latency:",
        round(
            result["latency"],
            4
        ),
        "seconds"
    )


# ============================================================
# CREATE APPLICATION CONFIGURATION
#
# This represents configuration that can later be moved to
# environment variables in a cloud deployment.
# ============================================================

APP_CONFIG = {

    "application_name":
        "Cloud-Ready LangChain AI Assistant",

    "version":
        "1.0",

    "environment":
        "development",

    "retrieval_top_k":
        3,

    "max_context_documents":
        3,

    "model_provider":
        "lightweight-local-demo",

    "storage_mode":
        "minimal",

    "device":
        "CPU"

}


print()
print("=" * 60)
print("APPLICATION CONFIGURATION")
print("=" * 60)


for key, value in APP_CONFIG.items():

    print(
        key,
        ":",
        value
    )


# ============================================================
# BASIC APPLICATION HEALTH CHECK
# ============================================================

def health_check():

    checks = {

        "knowledge_base":
            len(documents) > 0,

        "entity_retriever":
            callable(
                retrieve_documents
            ),

        "context_builder":
            callable(
                create_context
            ),

        "langchain_pipeline":
            callable(
                langchain_assistant
            )

    }


    overall_status = all(
        checks.values()
    )


    return {

        "status":
            "healthy"
            if overall_status
            else "unhealthy",

        "checks":
            checks

    }


health = health_check()


print()
print("=" * 60)
print("APPLICATION HEALTH CHECK")
print("=" * 60)

print()
print(
    "Status:",
    health["status"]
)


for check, status in health[
    "checks"
].items():

    print(
        check,
        ":",
        "PASS" if status else "FAIL"
    )


# ============================================================
# SIMPLE PERFORMANCE TEST
# ============================================================

performance_questions = [

    "What is LangChain?",

    "What is RAG?",

    "What are AI agents?",

    "What is Docker?",

    "What is FastAPI?"

]


latencies = []


print()
print("=" * 60)
print("PERFORMANCE TEST")
print("=" * 60)


for question in performance_questions:

    start = time.time()


    result = langchain_assistant(
        question
    )


    end = time.time()


    latency = end - start


    latencies.append(
        latency
    )


    print(
        round(
            latency,
            5
        ),
        "seconds →",
        question
    )


average_latency = (
    sum(latencies)
    / len(latencies)
)


print()
print(
    "Average latency:",
    round(
        average_latency,
        5
    ),
    "seconds"
)


# ============================================================
# PROJECT PIPELINE VISUALIZATION
# ============================================================

print()
print("=" * 60)
print("LANGCHAIN APPLICATION ARCHITECTURE")
print("=" * 60)

print("""
                    USER
                     |
                     v
              USER QUESTION
                     |
                     v
            LANGCHAIN PIPELINE
                     |
                     v
              DOCUMENT RETRIEVER
                     |
                     v
             RELEVANT CONTEXT
                     |
                     v
             PROMPT TEMPLATE
                     |
                     v
             RESPONSE ENGINE
                     |
                     v
              FINAL RESPONSE
""")


# ============================================================
# WHAT WE BUILT
# ============================================================

print()
print("=" * 60)
print("PART 1 — WHAT WE BUILT")
print("=" * 60)

print("""
1. Local knowledge base
2. LangChain Documents
3. Lightweight document retriever
4. Context builder
5. ChatPromptTemplate
6. LangChain Runnable components
7. Complete retrieval pipeline
8. Response generation
9. Application configuration
10. Health check
11. Performance measurement
""")


# ============================================================
# INTERVIEW EXPLANATION
# ============================================================

print()
print("=" * 60)
print("INTERVIEW EXPLANATION")
print("=" * 60)

print("""
I built a lightweight cloud-ready AI assistant using LangChain.

The application receives a user query and first retrieves
relevant information from a local knowledge base. The retrieved
information is converted into context and passed through a
LangChain pipeline before generating the final response.

I intentionally separated retrieval, context construction,
prompting, and response generation so that the response layer
can later be replaced with an external LLM API without
redesigning the complete application.

The current implementation is CPU and storage friendly because
it does not require a large local language model.
""")


# ============================================================
# FINAL VALIDATION
# ============================================================

print()
print("=" * 60)
print("DAY 56 — PART 1 COMPLETE ✔")
print("=" * 60)

print()
print("✔ Local knowledge base")
print("✔ LangChain Documents")
print("✔ Lightweight retrieval")
print("✔ Context generation")
print("✔ ChatPromptTemplate")
print("✔ Runnable pipeline")
print("✔ AI assistant")
print("✔ Health check")
print("✔ Performance test")
print("✔ Cloud-ready configuration")

print()
print("HARDWARE:")
print("✔ CPU friendly")
print("✔ Minimal storage")
print("✔ No GPU")
print("✔ No large LLM")
print("✔ No external dataset")

print()
print("NEXT → PART 2: LANGCHAIN TOOLS + AGENT WORKFLOW")


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\SaiVamshiGudem\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip
C:\Users\SaiVamshiGudem\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LangChain imported successfully ✔

Cloud-Ready LangChain AI Assistant
Day 56/100

Knowledge base created ✔
Number of documents: 7

KNOWLEDGE BASE

Document: 1
Topic: LangChain
Content: LangChain is a framework for building applications
        powered by language models. It provides components
        for prompts, document processing, retrieval, tools,
        agents, and application workflows.

Document: 2
Topic: RAG
Content: Retrieval-Augmented Generation, commonly called RAG,
        combines information retrieval with language models.
        A RAG system retrieves relevant information from a
        knowledge source and uses that information to generate
        a response.

Document: 3
Topic: AI Agents
Content: AI agents are systems that can decide which actions
        or tools should be used to solve a task. Agents can
        interact with tools, observe results, and continue
        processing until the task is completed.

Document: 4
Topic: FastAPI
Content: FastAPI is a Pytho

In [2]:
# ============================================================
# DAY 56/100
# PROJECT: CLOUD-READY LANGCHAIN AI ASSISTANT
# PART 2: LANGCHAIN TOOLS + AGENT WORKFLOW
#
# CONTINUES FROM PART 1
#
# JUPYTER NOTEBOOK ONLY
#
# HARDWARE:
# - SLOW CPU
# - VERY LIMITED STORAGE
# - NO GPU
# - NO LARGE LLM
# - NO EXTERNAL DATASET
# ============================================================


print("=" * 60)
print("DAY 56/100 — PART 2")
print("LANGCHAIN TOOLS + AGENT WORKFLOW")
print("=" * 60)


# ============================================================
# IMPORTS
# ============================================================

from langchain_core.tools import tool
from langchain_core.runnables import RunnableLambda

import re
import math
import time


print()
print("Part 2 libraries imported successfully ✔")


# ============================================================
# VERIFY PART 1
# ============================================================

required_variables = [

    "documents",
    "retrieve_documents",
    "create_context",
    "langchain_assistant",
    "APP_CONFIG"

]


missing_variables = []


for variable in required_variables:

    if variable not in globals():

        missing_variables.append(
            variable
        )


if len(missing_variables) > 0:

    print()
    print(
        "Missing Part 1 variables:"
    )

    print(
        missing_variables
    )

    print()
    print(
        "Please run Part 1 before Part 2."
    )

else:

    print()
    print(
        "Part 1 detected successfully ✔"
    )


# ============================================================
# TOOL 1 — CALCULATOR
#
# A LangChain tool that performs basic mathematical
# calculations.
#
# We intentionally avoid eval() for safety.
# ============================================================

@tool
def calculator(expression: str) -> str:
    """
    Perform basic mathematical calculations.
    Supports numbers, +, -, *, /, %, ** and parentheses.
    """

    try:

        expression = expression.strip()


        # ----------------------------------------------------
        # Allow only safe mathematical characters
        # ----------------------------------------------------

        if not re.fullmatch(
            r"[0-9+\-*/().%\s]+",
            expression
        ):

            return (
                "Calculator error: "
                "unsupported characters."
            )


        # ----------------------------------------------------
        # Convert percentage
        # Example:
        # 20 % 5
        # ----------------------------------------------------

        expression = expression.replace(
            "%",
            "/100"
        )


        # ----------------------------------------------------
        # Safe evaluation
        # ----------------------------------------------------

        allowed_names = {

            "__builtins__": {},

            "abs": abs,

            "round": round

        }


        result = eval(
            expression,
            allowed_names
        )


        return str(result)


    except Exception as error:

        return (
            "Calculator error: "
            + str(error)
        )


# ============================================================
# TEST CALCULATOR TOOL
# ============================================================

calculator_tests = [

    "10 + 20",

    "100 * 5",

    "100 / 4",

    "(10 + 5) * 2",

    "2 ** 5"

]


print()
print("=" * 60)
print("CALCULATOR TOOL TEST")
print("=" * 60)


for expression in calculator_tests:

    result = calculator.invoke(
        expression
    )


    print()
    print(
        expression,
        "=",
        result
    )


# ============================================================
# TOOL 2 — KNOWLEDGE SEARCH
#
# This tool uses the retriever created in Part 1.
# ============================================================

@tool
def knowledge_search(query: str) -> str:
    """
    Search the local AI knowledge base for relevant information.
    """

    try:

        retrieved_documents = retrieve_documents(
            query,
            top_k=3
        )


        if len(retrieved_documents) == 0:

            return (
                "No relevant information "
                "was found in the knowledge base."
            )


        context = create_context(
            retrieved_documents
        )


        return context


    except Exception as error:

        return (
            "Knowledge search error: "
            + str(error)
        )


# ============================================================
# TEST KNOWLEDGE SEARCH TOOL
# ============================================================

search_queries = [

    "What is LangChain?",

    "What is RAG?",

    "What is Docker?"

]


print()
print("=" * 60)
print("KNOWLEDGE SEARCH TOOL TEST")
print("=" * 60)


for query in search_queries:

    result = knowledge_search.invoke(
        query
    )


    print()
    print(
        "QUERY:",
        query
    )

    print()

    print(
        "RESULT:"
    )

    print(
        result
    )


# ============================================================
# TOOL 3 — APPLICATION STATUS
#
# This represents a simple production-style monitoring tool.
# ============================================================

@tool
def application_status(query: str = "") -> str:
    """
    Return the current status of the AI assistant application.
    """

    return (
        "Application Status: HEALTHY\n"
        "Version: "
        + APP_CONFIG["version"]
        + "\n"
        "Environment: "
        + APP_CONFIG["environment"]
        + "\n"
        "Device: "
        + APP_CONFIG["device"]
        + "\n"
        "Knowledge Documents: "
        + str(len(documents))
    )


# ============================================================
# TEST APPLICATION STATUS
# ============================================================

print()
print("=" * 60)
print("APPLICATION STATUS TOOL TEST")
print("=" * 60)

print()

print(
    application_status.invoke("")
)


# ============================================================
# REGISTER ALL TOOLS
# ============================================================

TOOLS = {

    "calculator":
        calculator,

    "knowledge_search":
        knowledge_search,

    "application_status":
        application_status

}


print()
print("=" * 60)
print("AVAILABLE TOOLS")
print("=" * 60)


for tool_name in TOOLS:

    print(
        "✔",
        tool_name
    )


# ============================================================
# TOOL DESCRIPTION
# ============================================================

print()
print("=" * 60)
print("TOOL DESCRIPTIONS")
print("=" * 60)


for tool_name, tool_object in TOOLS.items():

    print()

    print(
        "Tool:",
        tool_name
    )

    print(
        "Description:",
        tool_object.description
    )


# ============================================================
# SIMPLE INTENT DETECTION
#
# In a full production system, an LLM can decide which tool
# should be used.
#
# Because your machine has almost no storage and a slow CPU,
# we implement a lightweight deterministic router.
#
# The architecture remains the same:
#
# USER QUERY
#      ↓
# AGENT / ROUTER
#      ↓
# TOOL SELECTION
#      ↓
# TOOL EXECUTION
# ============================================================

def detect_intent(
    query
):

    query_lower = query.lower()


    # --------------------------------------------------------
    # CALCULATOR INTENT
    # --------------------------------------------------------

    math_keywords = [

        "calculate",

        "calculator",

        "what is",

        "compute",

        "multiply",

        "divide",

        "plus",

        "minus"

    ]


    # Check for explicit mathematical expressions

    contains_math_expression = bool(

        re.search(
            r"\d+\s*[\+\-\*/]\s*\d+",
            query
        )

    )


    if contains_math_expression:

        return "calculator"


    if any(
        keyword in query_lower
        for keyword in [
            "calculate",
            "compute",
            "multiply",
            "divide"
        ]
    ):

        return "calculator"


    # --------------------------------------------------------
    # APPLICATION STATUS INTENT
    # --------------------------------------------------------

    status_keywords = [

        "status",

        "health",

        "healthy",

        "application status",

        "system status"

    ]


    if any(
        keyword in query_lower
        for keyword in status_keywords
    ):

        return "application_status"


    # --------------------------------------------------------
    # KNOWLEDGE SEARCH INTENT
    # --------------------------------------------------------

    knowledge_keywords = [

        "what is",

        "what are",

        "explain",

        "tell me about",

        "how does",

        "how do",

        "define",

        "langchain",

        "rag",

        "docker",

        "fastapi",

        "agent",

        "cloud"

    ]


    if any(
        keyword in query_lower
        for keyword in knowledge_keywords
    ):

        return "knowledge_search"


    # --------------------------------------------------------
    # DEFAULT
    # --------------------------------------------------------

    return "knowledge_search"


# ============================================================
# TEST INTENT DETECTION
# ============================================================

intent_tests = [

    "What is LangChain?",

    "Calculate 25 * 4",

    "What is RAG?",

    "Show application health",

    "What is Docker?",

    "Compute 100 / 5"

]


print()
print("=" * 60)
print("INTENT DETECTION TEST")
print("=" * 60)


for query in intent_tests:

    intent = detect_intent(
        query
    )


    print()

    print(
        "Query:",
        query
    )

    print(
        "Selected Tool:",
        intent
    )


# ============================================================
# EXTRACT MATHEMATICAL EXPRESSION
# ============================================================

def extract_math_expression(
    query
):

    # --------------------------------------------------------
    # Search for a basic mathematical expression
    # --------------------------------------------------------

    match = re.search(

        r"(\d+(?:\.\d+)?\s*"
        r"(?:\+|\-|\*|/|\*\*)\s*"
        r"\d+(?:\.\d+)?)",

        query

    )


    if match:

        return match.group(
            1
        )


    # --------------------------------------------------------
    # Handle phrases such as:
    # "multiply 10 by 5"
    # --------------------------------------------------------

    multiply_match = re.search(

        r"multiply\s+"
        r"(\d+(?:\.\d+)?)"
        r"\s+(?:by|and)\s+"
        r"(\d+(?:\.\d+)?)",

        query.lower()

    )


    if multiply_match:

        first = multiply_match.group(
            1
        )

        second = multiply_match.group(
            2
        )

        return (
            first
            + " * "
            + second
        )


    return None


# ============================================================
# TEST MATH EXTRACTION
# ============================================================

print()
print("=" * 60)
print("MATH EXPRESSION EXTRACTION")
print("=" * 60)


math_queries = [

    "Calculate 25 * 4",

    "Compute 100 / 5",

    "What is 10 + 20?",

    "multiply 12 by 5"

]


for query in math_queries:

    expression = extract_math_expression(
        query
    )


    print()

    print(
        "Query:",
        query
    )

    print(
        "Expression:",
        expression
    )


# ============================================================
# TOOL EXECUTION ENGINE
#
# This is the component that executes the tool selected by
# the agent/router.
# ============================================================

def execute_tool(
    tool_name,
    query
):

    # --------------------------------------------------------
    # CALCULATOR
    # --------------------------------------------------------

    if tool_name == "calculator":

        expression = extract_math_expression(
            query
        )


        if expression is None:

            return (
                "I could not identify a "
                "mathematical expression."
            )


        return calculator.invoke(
            expression
        )


    # --------------------------------------------------------
    # KNOWLEDGE SEARCH
    # --------------------------------------------------------

    if tool_name == "knowledge_search":

        return knowledge_search.invoke(
            query
        )


    # --------------------------------------------------------
    # APPLICATION STATUS
    # --------------------------------------------------------

    if tool_name == "application_status":

        return application_status.invoke(
            query
        )


    # --------------------------------------------------------
    # UNKNOWN TOOL
    # --------------------------------------------------------

    return (
        "Unknown tool: "
        + str(tool_name)
    )


# ============================================================
# TEST TOOL EXECUTION
# ============================================================

execution_tests = [

    "What is LangChain?",

    "Calculate 50 * 2",

    "What is RAG?",

    "Show application status"

]


print()
print("=" * 60)
print("TOOL EXECUTION TEST")
print("=" * 60)


for query in execution_tests:

    selected_tool = detect_intent(
        query
    )


    result = execute_tool(
        selected_tool,
        query
    )


    print()
    print(
        "USER:",
        query
    )

    print(
        "TOOL:",
        selected_tool
    )

    print(
        "RESULT:",
        result
    )


# ============================================================
# AGENT STATE
#
# This represents the state passed through an agentic
# workflow.
# ============================================================

def create_agent_state(
    query
):

    return {

        "query":
        query,

        "selected_tool":
        None,

        "tool_input":
        query,

        "tool_output":
        None,

        "final_answer":
        None,

        "steps":
        []

    }


# ============================================================
# AGENT STEP 1 — ANALYZE QUERY
# ============================================================

def agent_analyze(
    state
):

    selected_tool = detect_intent(
        state["query"]
    )


    state["selected_tool"] = (
        selected_tool
    )


    state["steps"].append(
        "intent_detection"
    )


    return state


# ============================================================
# AGENT STEP 2 — EXECUTE TOOL
# ============================================================

def agent_execute(
    state
):

    tool_name = state[
        "selected_tool"
    ]


    tool_output = execute_tool(
        tool_name,
        state["query"]
    )


    state["tool_output"] = (
        tool_output
    )


    state["steps"].append(
        "tool_execution"
    )


    return state


# ============================================================
# AGENT STEP 3 — CREATE FINAL RESPONSE
# ============================================================

def agent_generate_response(
    state
):

    query = state[
        "query"
    ]


    tool_name = state[
        "selected_tool"
    ]


    tool_output = state[
        "tool_output"
    ]


    if tool_name == "calculator":

        final_answer = (

            "Calculation result: "
            + str(tool_output)

        )


    elif tool_name == "application_status":

        final_answer = (

            "Here is the current application status:\n\n"
            + str(tool_output)

        )


    elif tool_name == "knowledge_search":

        final_answer = (

            "Based on the knowledge base:\n\n"
            + str(tool_output)

        )


    else:

        final_answer = (
            "I could not process the request."
        )


    state["final_answer"] = (
        final_answer
    )


    state["steps"].append(
        "response_generation"
    )


    return state


# ============================================================
# BUILD LANGCHAIN AGENT WORKFLOW
#
# RunnableLambda allows us to compose the workflow directly
# inside the Jupyter Notebook.
# ============================================================

agent_workflow = (

    RunnableLambda(
        agent_analyze
    )

    | RunnableLambda(
        agent_execute
    )

    | RunnableLambda(
        agent_generate_response
    )

)


print()
print("=" * 60)
print("LANGCHAIN AGENT WORKFLOW CREATED")
print("=" * 60)

print()
print("""
Query
  ↓
Agent Analysis
  ↓
Tool Selection
  ↓
Tool Execution
  ↓
Response Generation
  ↓
Final Answer
""")


# ============================================================
# RUN AGENT
# ============================================================

def run_agent(
    query
):

    start_time = time.time()


    state = create_agent_state(
        query
    )


    final_state = agent_workflow.invoke(
        state
    )


    final_state["latency"] = (
        time.time()
        - start_time
    )


    return final_state


# ============================================================
# TEST AGENT
# ============================================================

agent_test_queries = [

    "What is LangChain?",

    "Calculate 25 * 8",

    "What is RAG?",

    "What is Docker?",

    "Show application health",

    "Compute 100 / 4"

]


print()
print("=" * 60)
print("AGENT WORKFLOW TEST")
print("=" * 60)


for query in agent_test_queries:

    result = run_agent(
        query
    )


    print()
    print("-" * 60)

    print(
        "USER:"
    )

    print(
        query
    )

    print()

    print(
        "SELECTED TOOL:"
    )

    print(
        result["selected_tool"]
    )

    print()

    print(
        "STEPS:"
    )

    print(
        result["steps"]
    )

    print()

    print(
        "ASSISTANT:"
    )

    print(
        result["final_answer"]
    )

    print()

    print(
        "LATENCY:",
        round(
            result["latency"],
            5
        ),
        "seconds"
    )


# ============================================================
# ADD CONVERSATION MEMORY
#
# Lightweight Python memory instead of a database.
# ============================================================

conversation_history = []


def add_to_memory(
    user_query,
    assistant_response
):

    conversation_history.append({

        "user":
        user_query,

        "assistant":
        assistant_response

    })


# ============================================================
# MEMORY-AWARE AGENT
# ============================================================

def memory_agent(
    query
):

    result = run_agent(
        query
    )


    answer = result[
        "final_answer"
    ]


    add_to_memory(
        query,
        answer
    )


    return result


# ============================================================
# TEST MEMORY
# ============================================================

print()
print("=" * 60)
print("CONVERSATION MEMORY TEST")
print("=" * 60)


memory_queries = [

    "What is LangChain?",

    "What is RAG?",

    "What is Docker?"

]


for query in memory_queries:

    result = memory_agent(
        query
    )


    print()
    print(
        "USER:",
        query
    )

    print(
        "ASSISTANT:",
        result["final_answer"]
    )


print()
print(
    "Conversation turns:",
    len(conversation_history)
)


# ============================================================
# DISPLAY MEMORY
# ============================================================

print()
print("=" * 60)
print("CONVERSATION HISTORY")
print("=" * 60)


for index, conversation in enumerate(
    conversation_history
):

    print()

    print(
        "Turn:",
        index + 1
    )

    print(
        "User:",
        conversation["user"]
    )

    print(
        "Assistant:",
        conversation["assistant"]
    )


# ============================================================
# AGENT OBSERVABILITY
#
# We track:
# - selected tool
# - steps
# - latency
# ============================================================

def inspect_agent_run(
    query
):

    result = run_agent(
        query
    )


    print()
    print("=" * 60)
    print("AGENT OBSERVABILITY")
    print("=" * 60)


    print()

    print(
        "Query:",
        result["query"]
    )


    print()

    print(
        "Selected Tool:",
        result["selected_tool"]
    )


    print()

    print(
        "Workflow Steps:"
    )

    for step_number, step in enumerate(
        result["steps"]
    ):

        print(
            step_number + 1,
            "→",
            step
        )


    print()

    print(
        "Tool Output:"
    )

    print(
        result["tool_output"]
    )


    print()

    print(
        "Final Answer:"
    )

    print(
        result["final_answer"]
    )


    print()

    print(
        "Latency:",
        round(
            result["latency"],
            5
        ),
        "seconds"
    )


    return result


# ============================================================
# OBSERVABILITY TEST
# ============================================================

inspect_agent_run(
    "Calculate 45 * 12"
)


# ============================================================
# AGENT PERFORMANCE TEST
# ============================================================

performance_queries = [

    "What is LangChain?",

    "What is RAG?",

    "Calculate 20 * 5",

    "What is Docker?",

    "Show application status"

]


agent_latencies = []


print()
print("=" * 60)
print("AGENT PERFORMANCE TEST")
print("=" * 60)


for query in performance_queries:

    result = run_agent(
        query
    )


    agent_latencies.append(
        result["latency"]
    )


    print()

    print(
        query,
        "→",
        result["selected_tool"],
        "→",
        round(
            result["latency"],
            5
        ),
        "seconds"
    )


average_agent_latency = (

    sum(agent_latencies)
    /
    len(agent_latencies)

)


print()
print(
    "Average Agent Latency:",
    round(
        average_agent_latency,
        5
    ),
    "seconds"
)


# ============================================================
# COMPLETE AGENT ARCHITECTURE
# ============================================================

print()
print("=" * 60)
print("DAY 56 — PART 2 ARCHITECTURE")
print("=" * 60)

print("""
                         USER
                           |
                           v
                     USER QUERY
                           |
                           v
                 ┌─────────────────┐
                 │  LANGCHAIN      │
                 │     AGENT       │
                 └────────┬────────┘
                          |
                    INTENT DETECTION
                          |
              ┌───────────┼───────────┐
              |           |           |
              v           v           v
        Calculator   Knowledge     App Status
                       Search
              |           |           |
              └───────────┼───────────┘
                          |
                          v
                   TOOL EXECUTION
                          |
                          v
                    TOOL OUTPUT
                          |
                          v
                 RESPONSE GENERATION
                          |
                          v
                     FINAL ANSWER
                          |
                          v
                  CONVERSATION MEMORY
""")


# ============================================================
# CHAIN VS AGENT
# ============================================================

print()
print("=" * 60)
print("CHAIN VS AGENT")
print("=" * 60)

print("""
CHAIN:

User
 ↓
Step 1
 ↓
Step 2
 ↓
Step 3
 ↓
Answer

The execution path is mostly predetermined.


AGENT:

User
 ↓
Agent
 ↓
Decide what is needed
 ↓
Select a tool
 ↓
Execute tool
 ↓
Observe result
 ↓
Generate answer


KEY DIFFERENCE:

A chain follows a predefined sequence.

An agent can dynamically decide which tool/action
should be used for a request.
""")


# ============================================================
# INDUSTRY USE CASES
# ============================================================

print()
print("=" * 60)
print("INDUSTRY USE CASES")
print("=" * 60)

print("""
1. Customer Support AI
   → Search knowledge base
   → Answer customer questions

2. Banking AI Assistant
   → Account information tool
   → Transaction tool
   → Policy search

3. Insurance AI Assistant
   → Policy search
   → Claims information
   → Premium calculation

4. Developer Copilot
   → Code search
   → Documentation search
   → Testing tools

5. Enterprise AI Assistant
   → Multiple internal tools
   → RAG
   → Agents
   → APIs
""")


# ============================================================
# PART 2 VALIDATION
# ============================================================

print()
print("=" * 60)
print("DAY 56 — PART 2 COMPLETE ✔")
print("=" * 60)

print()

print("✔ LangChain tools created")
print("✔ Calculator tool")
print("✔ Knowledge search tool")
print("✔ Application status tool")
print("✔ Tool registry")
print("✔ Intent detection")
print("✔ Tool selection")
print("✔ Tool execution")
print("✔ Agent state")
print("✔ Agent workflow")
print("✔ Conversation memory")
print("✔ Agent observability")
print("✔ Latency measurement")
print("✔ Industry use cases")

print()

print("HARDWARE OPTIMIZATION")
print("✔ Jupyter Notebook only")
print("✔ No large LLM")
print("✔ No GPU")
print("✔ No external dataset")
print("✔ No vector database")
print("✔ No heavy model")
print("✔ Minimal storage")
print("✔ Slow CPU friendly")

print()

print("DAY 56 PART 2 COMPLETE ✔")

print()
print("NEXT → PART 3: FASTAPI + PRODUCTION API LAYER")

DAY 56/100 — PART 2
LANGCHAIN TOOLS + AGENT WORKFLOW

Part 2 libraries imported successfully ✔

Part 1 detected successfully ✔

CALCULATOR TOOL TEST

10 + 20 = 30

100 * 5 = 500

100 / 4 = 25.0

(10 + 5) * 2 = 30

2 ** 5 = 32

KNOWLEDGE SEARCH TOOL TEST

QUERY: What is LangChain?

RESULT:
Topic: LangChain
LangChain is a framework for building applications
        powered by language models. It provides components
        for prompts, document processing, retrieval, tools,
        agents, and application workflows.

QUERY: What is RAG?

RESULT:
Topic: RAG
Retrieval-Augmented Generation, commonly called RAG,
        combines information retrieval with language models.
        A RAG system retrieves relevant information from a
        knowledge source and uses that information to generate
        a response.

QUERY: What is Docker?

RESULT:
Topic: Docker
Docker packages an application and its dependencies
        into a container. Containers help applications run
        consistently acro

In [3]:
# ============================================================
# DAY 56/100
# PROJECT: CLOUD-READY LANGCHAIN AI ASSISTANT
# PART 3: FASTAPI + PRODUCTION API LAYER
#
# CONTINUES FROM PART 1 + PART 2
#
# JUPYTER NOTEBOOK ONLY
#
# HARDWARE:
# - SLOW CPU
# - VERY LIMITED STORAGE
# - NO GPU
# - NO LARGE LLM
# ============================================================


# ============================================================
# INSTALL MINIMAL API PACKAGES
# ============================================================

!pip install -q fastapi uvicorn


# ============================================================
# IMPORTS
# ============================================================

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from fastapi.testclient import TestClient

import time
import uuid
import logging
import os


print("=" * 60)
print("DAY 56 — PART 3")
print("FASTAPI + PRODUCTION API LAYER")
print("=" * 60)

print()
print("FastAPI imported successfully ✔")


# ============================================================
# VERIFY PART 2
# ============================================================

required_variables = [

    "run_agent",
    "TOOLS",
    "conversation_history",
    "APP_CONFIG"

]


missing_variables = []


for variable in required_variables:

    if variable not in globals():

        missing_variables.append(
            variable
        )


if len(missing_variables) > 0:

    print()
    print(
        "Missing variables from Part 2:"
    )

    print(
        missing_variables
    )

    print()
    print(
        "Please run Part 1 and Part 2 first."
    )

else:

    print()
    print(
        "Part 2 detected successfully ✔"
    )


# ============================================================
# APPLICATION CONFIGURATION
# ============================================================

APP_NAME = (
    "Cloud-Ready LangChain AI Assistant"
)

APP_VERSION = "1.0.0"

ENVIRONMENT = "development"

API_PREFIX = "/api/v1"


print()
print("=" * 60)
print("APPLICATION CONFIGURATION")
print("=" * 60)

print(
    "Application:",
    APP_NAME
)

print(
    "Version:",
    APP_VERSION
)

print(
    "Environment:",
    ENVIRONMENT
)

print(
    "API Prefix:",
    API_PREFIX
)


# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(
    level=logging.INFO
)


logger = logging.getLogger(
    "langchain_ai_assistant"
)


logger.info(
    "Application logging initialized"
)


# ============================================================
# CREATE FASTAPI APPLICATION
# ============================================================

app = FastAPI(

    title=APP_NAME,

    version=APP_VERSION,

    description=(
        "Lightweight LangChain AI Assistant API"
    )

)


print()
print(
    "FastAPI application created ✔"
)


# ============================================================
# REQUEST MODEL
# ============================================================

class ChatRequest(BaseModel):

    message: str = Field(

        ...,

        min_length=1,

        max_length=1000,

        description=(
            "User message"
        )

    )

    session_id: str | None = Field(

        default=None,

        description=(
            "Optional conversation session ID"
        )

    )


# ============================================================
# RESPONSE MODEL
# ============================================================

class ChatResponse(BaseModel):

    success: bool

    session_id: str

    message: str

    answer: str

    selected_tool: str

    latency_seconds: float


# ============================================================
# HEALTH RESPONSE MODEL
# ============================================================

class HealthResponse(BaseModel):

    status: str

    application: str

    version: str

    environment: str

    knowledge_documents: int

    available_tools: int


# ============================================================
# SIMPLE API SESSION STORAGE
#
# For this lightweight notebook project we use an in-memory
# dictionary instead of Redis or a database.
# ============================================================

sessions = {}


# ============================================================
# SESSION CREATION
# ============================================================

def get_or_create_session(
    session_id=None
):

    if session_id is not None:

        if session_id in sessions:

            return session_id


    new_session_id = str(
        uuid.uuid4()
    )


    sessions[new_session_id] = []


    return new_session_id


# ============================================================
# CHAT PROCESSING FUNCTION
# ============================================================

def process_chat_request(
    message,
    session_id=None
):

    start_time = time.time()


    # --------------------------------------------------------
    # SESSION
    # --------------------------------------------------------

    session_id = get_or_create_session(
        session_id
    )


    # --------------------------------------------------------
    # RUN LANGCHAIN AGENT
    # --------------------------------------------------------

    result = run_agent(
        message
    )


    # --------------------------------------------------------
    # STORE CONVERSATION
    # --------------------------------------------------------

    sessions[session_id].append({

        "user":
        message,

        "assistant":
        result["final_answer"],

        "tool":
        result["selected_tool"],

        "timestamp":
        time.time()

    })


    # --------------------------------------------------------
    # LATENCY
    # --------------------------------------------------------

    latency = (
        time.time()
        - start_time
    )


    return {

        "success":
        True,

        "session_id":
        session_id,

        "message":
        message,

        "answer":
        result["final_answer"],

        "selected_tool":
        result["selected_tool"],

        "latency_seconds":
        round(
            latency,
            5
        )

    }


# ============================================================
# HEALTH ENDPOINT
# ============================================================

@app.get(
    "/health",
    response_model=HealthResponse
)
def health():

    return {

        "status":
        "healthy",

        "application":
        APP_NAME,

        "version":
        APP_VERSION,

        "environment":
        ENVIRONMENT,

        "knowledge_documents":
        len(documents),

        "available_tools":
        len(TOOLS)

    }


# ============================================================
# ROOT ENDPOINT
# ============================================================

@app.get("/")
def root():

    return {

        "application":
        APP_NAME,

        "version":
        APP_VERSION,

        "message":
        "LangChain AI Assistant API is running",

        "health":
        "/health"

    }


# ============================================================
# CHAT ENDPOINT
# ============================================================

@app.post(
    API_PREFIX + "/chat",
    response_model=ChatResponse
)
def chat(
    request: ChatRequest
):

    try:

        result = process_chat_request(

            message=request.message,

            session_id=request.session_id

        )


        return result


    except Exception as error:

        logger.exception(
            "Chat request failed"
        )


        raise HTTPException(

            status_code=500,

            detail=(
                "Unable to process "
                "the request."
            )

        )


# ============================================================
# SESSION HISTORY ENDPOINT
# ============================================================

@app.get(
    API_PREFIX + "/sessions/{session_id}"
)
def get_session(
    session_id: str
):

    if session_id not in sessions:

        raise HTTPException(

            status_code=404,

            detail="Session not found"

        )


    return {

        "session_id":
        session_id,

        "conversation":
        sessions[session_id]

    }


# ============================================================
# CLEAR SESSION ENDPOINT
# ============================================================

@app.delete(
    API_PREFIX + "/sessions/{session_id}"
)
def delete_session(
    session_id: str
):

    if session_id not in sessions:

        raise HTTPException(

            status_code=404,

            detail="Session not found"

        )


    del sessions[session_id]


    return {

        "success":
        True,

        "message":
        "Session deleted",

        "session_id":
        session_id

    }


# ============================================================
# API INFORMATION ENDPOINT
# ============================================================

@app.get(
    API_PREFIX + "/info"
)
def api_info():

    return {

        "application":
        APP_NAME,

        "version":
        APP_VERSION,

        "environment":
        ENVIRONMENT,

        "tools":
        list(TOOLS.keys()),

        "knowledge_documents":
        len(documents),

        "endpoints": [

            "GET /",

            "GET /health",

            "GET /api/v1/info",

            "POST /api/v1/chat",

            "GET /api/v1/sessions/{session_id}",

            "DELETE /api/v1/sessions/{session_id}"

        ]

    }


# ============================================================
# CREATE TEST CLIENT
#
# This allows us to test the FastAPI application directly
# inside Jupyter without starting an external server.
# ============================================================

client = TestClient(
    app
)


print()
print("=" * 60)
print("FASTAPI TEST CLIENT CREATED")
print("=" * 60)

print()
print(
    "API can now be tested directly "
    "inside the Jupyter Notebook ✔"
)


# ============================================================
# TEST ROOT ENDPOINT
# ============================================================

response = client.get("/")


print()
print("=" * 60)
print("TEST 1 — ROOT ENDPOINT")
print("=" * 60)

print()

print(
    "Status Code:",
    response.status_code
)

print()

print(
    "Response:"
)

print(
    response.json()
)


# ============================================================
# TEST HEALTH ENDPOINT
# ============================================================

response = client.get(
    "/health"
)


print()
print("=" * 60)
print("TEST 2 — HEALTH ENDPOINT")
print("=" * 60)

print()

print(
    "Status Code:",
    response.status_code
)

print()

print(
    response.json()
)


# ============================================================
# TEST API INFORMATION
# ============================================================

response = client.get(
    API_PREFIX + "/info"
)


print()
print("=" * 60)
print("TEST 3 — API INFORMATION")
print("=" * 60)

print()

print(
    "Status Code:",
    response.status_code
)

print()

print(
    response.json()
)


# ============================================================
# TEST CHAT API — KNOWLEDGE QUERY
# ============================================================

payload = {

    "message":
    "What is LangChain?"

}


response = client.post(

    API_PREFIX + "/chat",

    json=payload

)


print()
print("=" * 60)
print("TEST 4 — CHAT API")
print("=" * 60)

print()

print(
    "Status Code:",
    response.status_code
)

print()

chat_result = response.json()


print(
    "Response:"
)

print(
    chat_result
)


# ============================================================
# EXTRACT SESSION ID
# ============================================================

session_id = chat_result[
    "session_id"
]


print()
print(
    "Created Session ID:"
)

print(
    session_id
)


# ============================================================
# TEST CHAT WITH CALCULATOR TOOL
# ============================================================

payload = {

    "message":
    "Calculate 25 * 8",

    "session_id":
    session_id

}


response = client.post(

    API_PREFIX + "/chat",

    json=payload

)


print()
print("=" * 60)
print("TEST 5 — CHAT + CALCULATOR TOOL")
print("=" * 60)

print()

print(
    "Status Code:",
    response.status_code
)

print()

print(
    response.json()
)


# ============================================================
# TEST CHAT WITH KNOWLEDGE SEARCH
# ============================================================

payload = {

    "message":
    "What is RAG?",

    "session_id":
    session_id

}


response = client.post(

    API_PREFIX + "/chat",

    json=payload

)


print()
print("=" * 60)
print("TEST 6 — CHAT + KNOWLEDGE SEARCH")
print("=" * 60)

print()

print(
    response.json()
)


# ============================================================
# TEST APPLICATION STATUS TOOL
# ============================================================

payload = {

    "message":
    "Show application health",

    "session_id":
    session_id

}


response = client.post(

    API_PREFIX + "/chat",

    json=payload

)


print()
print("=" * 60)
print("TEST 7 — APPLICATION STATUS")
print("=" * 60)

print()

print(
    response.json()
)


# ============================================================
# GET SESSION HISTORY
# ============================================================

response = client.get(

    API_PREFIX
    + "/sessions/"
    + session_id

)


print()
print("=" * 60)
print("TEST 8 — SESSION HISTORY")
print("=" * 60)

print()

print(
    "Status Code:",
    response.status_code
)

print()

session_result = response.json()


print(
    session_result
)


# ============================================================
# COUNT SESSION MESSAGES
# ============================================================

conversation_count = len(
    session_result[
        "conversation"
    ]
)


print()
print(
    "Conversation turns:",
    conversation_count
)


# ============================================================
# VALIDATION TEST — EMPTY MESSAGE
# ============================================================

invalid_payload = {

    "message":
    ""

}


response = client.post(

    API_PREFIX + "/chat",

    json=invalid_payload

)


print()
print("=" * 60)
print("TEST 9 — INPUT VALIDATION")
print("=" * 60)

print()

print(
    "Status Code:",
    response.status_code
)

print()

print(
    "Response:"
)

print(
    response.json()
)


# ============================================================
# VALIDATION TEST — VERY LARGE MESSAGE
# ============================================================

large_message = (
    "A" * 1200
)


invalid_payload = {

    "message":
    large_message

}


response = client.post(

    API_PREFIX + "/chat",

    json=invalid_payload

)


print()
print("=" * 60)
print("TEST 10 — MESSAGE SIZE VALIDATION")
print("=" * 60)

print()

print(
    "Status Code:",
    response.status_code
)

print()

print(
    response.json()
)


# ============================================================
# TEST MULTIPLE API REQUESTS
# ============================================================

test_requests = [

    "What is LangChain?",

    "What is RAG?",

    "What are AI agents?",

    "What is Docker?",

    "Calculate 50 * 2",

    "What is FastAPI?"

]


api_results = []


print()
print("=" * 60)
print("MULTI-REQUEST API TEST")
print("=" * 60)


for message in test_requests:

    start = time.time()


    response = client.post(

        API_PREFIX + "/chat",

        json={
            "message": message
        }

    )


    latency = (
        time.time()
        - start
    )


    result = response.json()


    api_results.append({

        "message":
        message,

        "status_code":
        response.status_code,

        "tool":
        result.get(
            "selected_tool"
        ),

        "latency":
        latency

    })


    print()

    print(
        "Message:",
        message
    )

    print(
        "Status:",
        response.status_code
    )

    print(
        "Tool:",
        result.get(
            "selected_tool"
        )
    )

    print(
        "Latency:",
        round(
            latency,
            5
        ),
        "seconds"
    )


# ============================================================
# API PERFORMANCE
# ============================================================

api_latencies = [

    result["latency"]

    for result in api_results

]


average_api_latency = (

    sum(api_latencies)
    /
    len(api_latencies)

)


successful_requests = sum(

    1

    for result in api_results

    if result["status_code"] == 200

)


success_rate = (

    successful_requests
    /
    len(api_results)
    * 100

)


print()
print("=" * 60)
print("API PERFORMANCE")
print("=" * 60)

print()

print(
    "Total Requests:",
    len(api_results)
)

print(
    "Successful Requests:",
    successful_requests
)

print(
    "Success Rate:",
    round(
        success_rate,
        2
    ),
    "%"
)

print(
    "Average Latency:",
    round(
        average_api_latency,
        5
    ),
    "seconds"
)


# ============================================================
# API ENDPOINT SUMMARY
# ============================================================

print()
print("=" * 60)
print("API ENDPOINTS")
print("=" * 60)

print("""
GET     /
GET     /health
GET     /api/v1/info
POST    /api/v1/chat
GET     /api/v1/sessions/{session_id}
DELETE  /api/v1/sessions/{session_id}
""")


# ============================================================
# PRODUCTION CONFIGURATION
#
# These values are kept simple for the notebook.
# In a real cloud environment, sensitive values should come
# from environment variables or a secret manager.
# ============================================================

PRODUCTION_CONFIG = {

    "environment":
        "production",

    "host":
        "0.0.0.0",

    "port":
        8000,

    "workers":
        1,

    "log_level":
        "info",

    "max_request_length":
        1000,

    "storage":
        "in-memory",

    "llm_mode":
        "replaceable"

}


print()
print("=" * 60)
print("PRODUCTION CONFIGURATION")
print("=" * 60)


for key, value in PRODUCTION_CONFIG.items():

    print(
        key,
        ":",
        value
    )


# ============================================================
# ENVIRONMENT VARIABLE DEMONSTRATION
# ============================================================

APP_ENV = os.getenv(
    "APP_ENV",
    "development"
)


LOG_LEVEL = os.getenv(
    "LOG_LEVEL",
    "INFO"
)


print()
print("=" * 60)
print("ENVIRONMENT VARIABLES")
print("=" * 60)

print(
    "APP_ENV:",
    APP_ENV
)

print(
    "LOG_LEVEL:",
    LOG_LEVEL
)


# ============================================================
# SIMPLE API MONITORING
# ============================================================

monitoring = {

    "total_requests":
        len(api_results),

    "successful_requests":
        successful_requests,

    "failed_requests":
        len(api_results)
        - successful_requests,

    "success_rate":
        round(
            success_rate,
            2
        ),

    "average_latency":
        round(
            average_api_latency,
            5
        )

}


print()
print("=" * 60)
print("MONITORING METRICS")
print("=" * 60)


for metric, value in monitoring.items():

    print(
        metric,
        ":",
        value
    )


# ============================================================
# GENERATE CLOUD DEPLOYMENT COMMAND
#
# We don't actually start a server here because the project
# must remain notebook-only.
#
# This shows the command that would be used when the notebook
# application is converted into a deployable service.
# ============================================================

deployment_command = (
    "uvicorn app:app "
    "--host 0.0.0.0 "
    "--port 8000"
)


print()
print("=" * 60)
print("CLOUD DEPLOYMENT COMMAND")
print("=" * 60)

print()

print(
    deployment_command
)


# ============================================================
# PRODUCTION ARCHITECTURE
# ============================================================

print()
print("=" * 60)
print("PART 3 — PRODUCTION ARCHITECTURE")
print("=" * 60)

print("""
                    USER / CLIENT
                          |
                          v
                   HTTP REQUEST
                          |
                          v
                 ┌────────────────┐
                 │    FastAPI     │
                 │   API Layer    │
                 └───────┬────────┘
                         |
                         v
                 ┌────────────────┐
                 │   LangChain    │
                 │     Agent      │
                 └───────┬────────┘
                         |
              ┌──────────┼──────────┐
              |          |          |
              v          v          v
         Calculator   Knowledge   App Status
                       Search
              |          |          |
              └──────────┼──────────┘
                         |
                         v
                  Final Response
                         |
                         v
                    FastAPI
                         |
                         v
                  JSON Response
                         |
                         v
                       USER
""")


# ============================================================
# INTERVIEW EXPLANATION
# ============================================================

print()
print("=" * 60)
print("INTERVIEW EXPLANATION")
print("=" * 60)

print("""
I exposed my LangChain AI assistant through a FastAPI
REST API.

The API contains a health endpoint, an information endpoint,
a chat endpoint, and session-management endpoints.

When a chat request arrives, FastAPI validates the request,
creates or retrieves a session, sends the message to the
LangChain agent, executes the appropriate tool, and returns
a structured JSON response.

I also added input validation, error handling, logging,
session management, latency measurement, and basic
monitoring metrics.

For this lightweight implementation I used in-memory session
storage. In production, this can be replaced with Redis or
another persistent store.

The architecture is cloud-ready because the API layer is
separated from the LangChain agent and tool layer.
""")


# ============================================================
# PRODUCTION CONSIDERATIONS
# ============================================================

print()
print("=" * 60)
print("PRODUCTION CONSIDERATIONS")
print("=" * 60)

print("""
Current Notebook:
    In-memory sessions
    Lightweight router
    Local knowledge base
    CPU-friendly implementation

Production:
    Redis / database for sessions
    Hosted LLM API
    Vector database for RAG
    Authentication / authorization
    Rate limiting
    HTTPS
    Secret manager
    Monitoring
    Distributed logging
    Horizontal scaling
    Container deployment
""")


# ============================================================
# PART 3 FINAL VALIDATION
# ============================================================

print()
print("=" * 60)
print("DAY 56 — PART 3 COMPLETE ✔")
print("=" * 60)

print()

print("✔ FastAPI application")
print("✔ Request model")
print("✔ Response model")
print("✔ Health endpoint")
print("✔ Chat endpoint")
print("✔ Session management")
print("✔ API information endpoint")
print("✔ Input validation")
print("✔ Error handling")
print("✔ Logging")
print("✔ Environment configuration")
print("✔ API testing")
print("✔ Performance testing")
print("✔ Monitoring metrics")
print("✔ Cloud deployment command")
print("✔ Production architecture")

print()

print("JUPYTER NOTEBOOK ONLY ✔")
print("CPU FRIENDLY ✔")
print("LOW STORAGE ✔")
print("NO GPU ✔")
print("NO LARGE LLM ✔")

print()

print("NEXT → PART 4: FINAL CLOUD-READY DEPLOYMENT")


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\SaiVamshiGudem\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip
INFO:langchain_ai_assistant:Application logging initialized
INFO:httpx:HTTP Request: GET http://testserver/ "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://testserver/api/v1/info "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://testserver/api/v1/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://testserver/api/v1/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://testserver/api/v1/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://testserver/api/v1/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://testserver/api/v1/sessions/711f3209-aa97-4855-855a-dba5ccbd9969 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://testserver/api/v1/chat "HTTP/1.1 422 Unprocessable Entity"
INFO:httpx

DAY 56 — PART 3
FASTAPI + PRODUCTION API LAYER

FastAPI imported successfully ✔

Part 2 detected successfully ✔

APPLICATION CONFIGURATION
Application: Cloud-Ready LangChain AI Assistant
Version: 1.0.0
Environment: development
API Prefix: /api/v1

FastAPI application created ✔

FASTAPI TEST CLIENT CREATED

API can now be tested directly inside the Jupyter Notebook ✔

TEST 1 — ROOT ENDPOINT

Status Code: 200

Response:
{'application': 'Cloud-Ready LangChain AI Assistant', 'version': '1.0.0', 'message': 'LangChain AI Assistant API is running', 'health': '/health'}

TEST 2 — HEALTH ENDPOINT

Status Code: 200

{'status': 'healthy', 'application': 'Cloud-Ready LangChain AI Assistant', 'version': '1.0.0', 'environment': 'development', 'knowledge_documents': 7, 'available_tools': 3}

TEST 3 — API INFORMATION

Status Code: 200

{'application': 'Cloud-Ready LangChain AI Assistant', 'version': '1.0.0', 'environment': 'development', 'tools': ['calculator', 'knowledge_search', 'application_status'

In [4]:
# ============================================================
# DAY 56/100 — CLOUD-READY LANGCHAIN AI ASSISTANT
# PART 4 — FINAL PRODUCTION + CLOUD READY VERSION
# ============================================================

print("=" * 70)
print("DAY 56/100 — CLOUD-READY LANGCHAIN AI ASSISTANT")
print("PART 4 — FINAL PRODUCTION + CLOUD READINESS")
print("=" * 70)


# ============================================================
# 1. VERIFY PREVIOUS PARTS
# ============================================================

required_variables = [
    "app",
    "run_agent",
    "sessions",
    "APP_NAME",
    "APP_VERSION",
    "API_PREFIX",
    "TestClient"
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise RuntimeError(
        f"Missing variables from previous parts: {missing_variables}\n"
        "Please run Parts 1, 2 and 3 first."
    )

print("✅ Parts 1, 2 and 3 are available")
print("✅ FastAPI application detected")
print("✅ LangChain agent detected")
print("✅ Session memory detected")


# ============================================================
# 2. FINAL PRODUCTION CONFIGURATION
# ============================================================

import os
import time
import json
import statistics
from datetime import datetime


PRODUCTION_CONFIG = {
    "application_name": APP_NAME,
    "version": APP_VERSION,
    "environment": "production",

    # API
    "host": "0.0.0.0",
    "port": 8000,

    # Performance
    "workers": 1,
    "request_timeout_seconds": 30,

    # Limits
    "max_message_length": 1000,
    "max_session_history": 20,

    # AI configuration
    "retrieval_top_k": 3,

    # Monitoring
    "logging_enabled": True,
    "health_check_enabled": True,

    # Security
    "debug": False
}


print("\nPRODUCTION CONFIGURATION")
print("-" * 50)

for key, value in PRODUCTION_CONFIG.items():
    print(f"{key:30} : {value}")


# ============================================================
# 3. ENVIRONMENT CONFIGURATION
# ============================================================

# In real cloud deployment, these values should come
# from environment variables.

CLOUD_ENVIRONMENT = {
    "APP_ENV": os.getenv("APP_ENV", "production"),
    "APP_HOST": os.getenv("APP_HOST", "0.0.0.0"),
    "APP_PORT": os.getenv("APP_PORT", "8000"),
    "LOG_LEVEL": os.getenv("LOG_LEVEL", "INFO")
}


print("\nCLOUD ENVIRONMENT")
print("-" * 50)

for key, value in CLOUD_ENVIRONMENT.items():
    print(f"{key:20} : {value}")


# ============================================================
# 4. PRODUCTION HEALTH CHECK
# ============================================================

def production_health_check():
    """
    Lightweight production health check.

    Checks:
    1. FastAPI application
    2. LangChain agent
    3. Knowledge base
    4. Session memory
    """

    checks = {}

    # FastAPI
    checks["fastapi"] = app is not None

    # Agent
    checks["langchain_agent"] = callable(run_agent)

    # Knowledge base
    checks["knowledge_base"] = (
        "documents" in globals()
        and len(documents) > 0
    )

    # Session storage
    checks["session_memory"] = isinstance(sessions, dict)

    overall_status = all(checks.values())

    return {
        "status": "healthy" if overall_status else "degraded",
        "checks": checks,
        "timestamp": datetime.utcnow().isoformat()
    }


health_result = production_health_check()

print("\nHEALTH CHECK")
print("-" * 50)

print(json.dumps(health_result, indent=2))


# ============================================================
# 5. CLOUD-READY REQUEST PROCESSOR
# ============================================================

def cloud_chat(message, session_id=None):
    """
    Cloud-ready wrapper around the LangChain agent.

    This function:
    - validates request
    - executes agent
    - measures latency
    - returns structured response
    """

    start_time = time.perf_counter()

    # -------------------------
    # Input validation
    # -------------------------

    if not isinstance(message, str):
        raise TypeError("message must be a string")

    message = message.strip()

    if not message:
        raise ValueError("message cannot be empty")

    if len(message) > PRODUCTION_CONFIG["max_message_length"]:
        raise ValueError(
            f"message exceeds "
            f"{PRODUCTION_CONFIG['max_message_length']} characters"
        )

    # -------------------------
    # Agent execution
    # -------------------------

    result = run_agent(message)

    latency = time.perf_counter() - start_time

    return {
        "success": True,
        "session_id": session_id,
        "message": message,
        "answer": result.get("answer"),
        "selected_tool": result.get("selected_tool"),
        "latency_seconds": round(latency, 6),
        "environment": CLOUD_ENVIRONMENT["APP_ENV"]
    }


# ============================================================
# 6. END-TO-END CLOUD TEST
# ============================================================

print("\n" + "=" * 70)
print("END-TO-END CLOUD TEST")
print("=" * 70)


test_queries = [
    "What is LangChain?",
    "What is RAG?",
    "Calculate 25 * 4 + 10",
    "Is the application running?"
]


cloud_results = []

for query in test_queries:

    result = cloud_chat(query)

    cloud_results.append(result)

    print("\nQuery:", query)
    print("Tool:", result["selected_tool"])
    print("Answer:", result["answer"])
    print("Latency:", result["latency_seconds"], "seconds")


# ============================================================
# 7. PERFORMANCE EVALUATION
# ============================================================

latencies = [
    result["latency_seconds"]
    for result in cloud_results
]

average_latency = statistics.mean(latencies)
minimum_latency = min(latencies)
maximum_latency = max(latencies)

performance_report = {
    "total_requests": len(latencies),
    "average_latency_seconds": round(average_latency, 6),
    "minimum_latency_seconds": round(minimum_latency, 6),
    "maximum_latency_seconds": round(maximum_latency, 6)
}


print("\n" + "=" * 70)
print("PERFORMANCE REPORT")
print("=" * 70)

print(json.dumps(performance_report, indent=2))


# ============================================================
# 8. SIMPLE LOAD TEST
# ============================================================

print("\n" + "=" * 70)
print("LIGHTWEIGHT LOAD TEST")
print("=" * 70)

load_queries = [
    "What is LangChain?",
    "What is RAG?",
    "What is an AI agent?",
    "Calculate 10 + 20",
    "Is the application running?"
]

load_latencies = []

for i in range(10):

    query = load_queries[i % len(load_queries)]

    start = time.perf_counter()

    cloud_chat(query)

    latency = time.perf_counter() - start

    load_latencies.append(latency)


load_report = {
    "requests": len(load_latencies),
    "average_latency": round(
        statistics.mean(load_latencies), 6
    ),
    "p95_approx": round(
        sorted(load_latencies)[
            min(len(load_latencies) - 1, int(len(load_latencies) * 0.95))
        ],
        6
    ),
    "maximum_latency": round(
        max(load_latencies), 6
    )
}


print(json.dumps(load_report, indent=2))


# ============================================================
# 9. PRODUCTION METRICS
# ============================================================

PRODUCTION_METRICS = {
    "requests_processed": len(load_latencies),
    "successful_requests": len(load_latencies),
    "failed_requests": 0,

    "average_latency_seconds":
        round(statistics.mean(load_latencies), 6),

    "max_latency_seconds":
        round(max(load_latencies), 6),

    "knowledge_base_documents":
        len(documents),

    "active_sessions":
        len(sessions)
}


print("\n" + "=" * 70)
print("PRODUCTION METRICS")
print("=" * 70)

for key, value in PRODUCTION_METRICS.items():
    print(f"{key:30} : {value}")


# ============================================================
# 10. CLOUD DEPLOYMENT ARCHITECTURE
# ============================================================

cloud_architecture = """
                    ┌─────────────────────┐
                    │       USER          │
                    │ Web / Mobile / API  │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │    Cloud / HTTPS    │
                    │   Public Endpoint   │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │      FastAPI        │
                    │    API Layer        │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │  LangChain Agent    │
                    │   Decision Layer    │
                    └───────┬─────┬───────┘
                            │     │
                ┌───────────┘     └───────────┐
                ▼                             ▼
       ┌────────────────┐            ┌────────────────┐
       │ Knowledge Tool │            │ Calculator     │
       └───────┬────────┘            │ / Status Tool │
               │                     └────────────────┘
               ▼
       ┌────────────────┐
       │ Knowledge Base │
       │ Local / Vector │
       │ DB in future   │
       └────────────────┘
"""


print("\n" + "=" * 70)
print("CLOUD ARCHITECTURE")
print("=" * 70)

print(cloud_architecture)


# ============================================================
# 11. DOCKERFILE TEMPLATE
# ============================================================

dockerfile = """
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
""".strip()


print("\n" + "=" * 70)
print("DOCKERFILE TEMPLATE")
print("=" * 70)

print(dockerfile)


# ============================================================
# 12. REQUIREMENTS.TXT
# ============================================================

requirements_txt = """
langchain
langchain-core
fastapi
uvicorn
pydantic
""".strip()


print("\n" + "=" * 70)
print("REQUIREMENTS.TXT")
print("=" * 70)

print(requirements_txt)


# ============================================================
# 13. DOCKERIGNORE
# ============================================================

dockerignore = """
__pycache__/
*.pyc
.ipynb_checkpoints/
.env
.git/
.gitignore
README.md
*.log
""".strip()


print("\n" + "=" * 70)
print("DOCKERIGNORE")
print("=" * 70)

print(dockerignore)


# ============================================================
# 14. CLOUD START COMMAND
# ============================================================

START_COMMAND = (
    "uvicorn app:app "
    "--host 0.0.0.0 "
    "--port 8000"
)

print("\n" + "=" * 70)
print("CLOUD START COMMAND")
print("=" * 70)

print(START_COMMAND)


# ============================================================
# 15. NOTEBOOK-ONLY DEPLOYMENT CONFIGURATION
# ============================================================

NOTEBOOK_DEPLOYMENT_CONFIG = {
    "runtime": "Python 3.11",
    "server": "Uvicorn",
    "framework": "FastAPI",
    "ai_framework": "LangChain",
    "deployment_type": "Container / Cloud VM / Managed App Service",
    "port": 8000,
    "host": "0.0.0.0",
    "health_endpoint": "/health",
    "chat_endpoint": "/api/v1/chat",
    "environment": "production"
}


print("\n" + "=" * 70)
print("DEPLOYMENT CONFIGURATION")
print("=" * 70)

print(json.dumps(
    NOTEBOOK_DEPLOYMENT_CONFIG,
    indent=2
))


# ============================================================
# 16. API CONTRACT
# ============================================================

API_CONTRACT = {
    "GET /": "Application information",
    "GET /health": "Health check",
    "GET /api/v1/info": "Application metadata",
    "POST /api/v1/chat": "Send message to AI assistant",
    "GET /api/v1/sessions/{session_id}":
        "Get conversation history",
    "DELETE /api/v1/sessions/{session_id}":
        "Delete conversation"
}


print("\n" + "=" * 70)
print("API CONTRACT")
print("=" * 70)

for endpoint, description in API_CONTRACT.items():
    print(f"{endpoint:45} -> {description}")


# ============================================================
# 17. FINAL FASTAPI TEST
# ============================================================

print("\n" + "=" * 70)
print("FINAL FASTAPI TEST")
print("=" * 70)

client = TestClient(app)


# Root
response = client.get("/")

print("\nROOT")
print("Status:", response.status_code)
print(response.json())


# Health
response = client.get("/health")

print("\nHEALTH")
print("Status:", response.status_code)
print(response.json())


# Info
response = client.get("/api/v1/info")

print("\nINFO")
print("Status:", response.status_code)
print(response.json())


# Chat
response = client.post(
    "/api/v1/chat",
    json={
        "message": "What is RAG?"
    }
)

print("\nCHAT")
print("Status:", response.status_code)
print(response.json())


# ============================================================
# 18. FINAL PRODUCTION READINESS SCORE
# ============================================================

readiness_checks = {
    "LangChain core pipeline": True,
    "Knowledge retrieval": True,
    "Tool execution": True,
    "Agent routing": True,
    "Conversation memory": True,
    "FastAPI API": True,
    "Health endpoint": True,
    "Input validation": True,
    "Error handling": True,
    "Logging": True,
    "Environment configuration": True,
    "Performance measurement": True,
    "Docker configuration": True,
    "Cloud architecture": True
}


completed = sum(readiness_checks.values())
total = len(readiness_checks)

readiness_percentage = (
    completed / total
) * 100


print("\n" + "=" * 70)
print("PRODUCTION READINESS SCORE")
print("=" * 70)

for check, status in readiness_checks.items():
    print(
        f"{'✅' if status else '❌'} "
        f"{check}"
    )

print("\nCompleted:", completed, "/", total)
print("Readiness:", f"{readiness_percentage:.1f}%")


# ============================================================
# 19. FINAL PROJECT SUMMARY
# ============================================================

FINAL_PROJECT = {
    "project": "Cloud-Ready LangChain AI Assistant",

    "day": "56/100",

    "technologies": [
        "Python",
        "LangChain",
        "LangChain Core",
        "FastAPI",
        "Pydantic",
        "Uvicorn",
        "REST API",
        "Docker",
        "Cloud-ready architecture"
    ],

    "capabilities": [
        "Knowledge retrieval",
        "Tool routing",
        "Calculator tool",
        "Application status tool",
        "Conversation memory",
        "REST API",
        "Health monitoring",
        "Performance monitoring",
        "Production configuration",
        "Cloud deployment architecture"
    ]
}


print("\n" + "=" * 70)
print("FINAL PROJECT")
print("=" * 70)

print(json.dumps(
    FINAL_PROJECT,
    indent=2
))


# ============================================================
# 20. FINAL ARCHITECTURE EXPLANATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL END-TO-END FLOW")
print("=" * 70)

print("""
1. User sends a request
        ↓
2. FastAPI receives the request
        ↓
3. Request validation happens
        ↓
4. LangChain agent analyzes the request
        ↓
5. Agent selects the appropriate tool
        ↓
6. Tool executes
        ↓
7. Knowledge / calculation / status result is produced
        ↓
8. Agent generates the final answer
        ↓
9. FastAPI returns structured JSON
        ↓
10. Monitoring records latency and request metrics
""")


# ============================================================
# 21. INTERVIEW EXPLANATION
# ============================================================

interview_answer = """
I built a cloud-ready AI assistant using LangChain and FastAPI.

The system exposes a REST API through FastAPI. When a user sends
a request, the request is validated and passed to a LangChain-based
agent workflow.

The agent determines the user's intent and selects the appropriate
tool, such as knowledge retrieval, calculator, or application
status.

For knowledge-related questions, the system retrieves relevant
documents from the knowledge base and generates a contextual answer.

I also added conversation session management, health checks,
logging, latency monitoring, environment-based configuration,
and API validation.

The application is designed to be containerized using Docker and
deployed to a cloud environment behind a public HTTPS endpoint.

For production scale, I would replace the lightweight in-memory
components with a vector database, persistent session storage,
external LLM provider, authentication, centralized observability,
and horizontal scaling.
"""


print("\n" + "=" * 70)
print("INTERVIEW ANSWER")
print("=" * 70)

print(interview_answer)


# ============================================================
# 22. FINAL VALIDATION
# ============================================================

final_validation = {
    "application_available": app is not None,
    "agent_available": callable(run_agent),
    "knowledge_base_available": len(documents) > 0,
    "fastapi_available": True,
    "health_check": health_result["status"] == "healthy",
    "end_to_end_test": len(cloud_results) == len(test_queries),
    "performance_test": len(load_latencies) == 10,
    "production_config": len(PRODUCTION_CONFIG) > 0,
    "docker_template": len(dockerfile) > 0,
    "cloud_architecture": len(cloud_architecture) > 0
}


print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

for check, status in final_validation.items():
    print(
        f"{'✅ PASS' if status else '❌ FAIL'} "
        f"{check}"
    )


if all(final_validation.values()):
    print("\n🎉 PART 4 COMPLETED SUCCESSFULLY!")
    print("🚀 DAY 56/100 — CLOUD-READY LANGCHAIN AI ASSISTANT COMPLETE!")
else:
    print("\n⚠️ Some validation checks require attention.")

INFO:httpx:HTTP Request: GET http://testserver/ "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://testserver/api/v1/info "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://testserver/api/v1/chat "HTTP/1.1 200 OK"


DAY 56/100 — CLOUD-READY LANGCHAIN AI ASSISTANT
PART 4 — FINAL PRODUCTION + CLOUD READINESS
✅ Parts 1, 2 and 3 are available
✅ FastAPI application detected
✅ LangChain agent detected
✅ Session memory detected

PRODUCTION CONFIGURATION
--------------------------------------------------
application_name               : Cloud-Ready LangChain AI Assistant
version                        : 1.0.0
environment                    : production
host                           : 0.0.0.0
port                           : 8000
workers                        : 1
request_timeout_seconds        : 30
max_message_length             : 1000
max_session_history            : 20
retrieval_top_k                : 3
logging_enabled                : True
health_check_enabled           : True
debug                          : False

CLOUD ENVIRONMENT
--------------------------------------------------
APP_ENV              : production
APP_HOST             : 0.0.0.0
APP_PORT             : 8000
LOG_LEVEL            : INF